In [2]:
import os
import io
import requests
import pandas as pd
from bs4 import BeautifulSoup

# 1. 초기 설정
url = "https://www.sphericalinsights.com/ko/reports/japan-k-beauty-product-market"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
PROJECT_NAME = "Spherical_Insights_Result"
os.makedirs(PROJECT_NAME, exist_ok=True)

def flatten_cols(df):
    """표의 헤더가 지저분할 경우 한 줄로 정리"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in col if 'Unnamed' not in str(c)]).strip() for col in df.columns.values]
    return df

try:
    # 2. 페이지 접속
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')

    # 3. 타겟 구역 추출 (div.tabs-container.desc-body)
    # 클래스명이 여러 개일 경우 마침표(.)로 연결합니다.
    target_area = soup.select_one('div.tabs-container.desc-body')

    if target_area:
        print("✅ 타겟 컨텐츠 영역을 찾았습니다.")
        
        final_content = [f"SOURCE_URL: {url}\n"]
        table_md_list = []

        # 4. 표(Table) 추출 로직
        # table-responsive-outer 클래스 내부의 모든 테이블을 찾습니다.
        table_wrappers = target_area.select('div.table-responsive-outer')
        
        for t_idx, wrapper in enumerate(table_wrappers, 1):
            table_tag = wrapper.find('table')
            if table_tag:
                try:
                    # 판다스로 표 읽기
                    df_list = pd.read_html(io.StringIO(str(table_tag)))
                    if df_list:
                        df = flatten_cols(df_list[0])
                        # 마크다운 변환
                        md_table = df.to_markdown(index=False)
                        table_md_list.append(f"\n[데이터 표 {t_idx}]\n{md_table}\n")
                        
                        # [중요] 본문 텍스트 추출 시 중복 방지를 위해 표 요소를 제거
                        wrapper.decompose() 
                except Exception as e:
                    print(f"  ⚠️ 표 {t_idx} 변환 실패: {e}")

        # 5. 본문 텍스트 추출 (표가 제거된 상태)
        cleaned_text = target_area.get_text(separator='\n', strip=True)

        # 6. 결과 조립 및 저장
        final_content.append("\n### [STRUCTURED TABLES]")
        final_content.append("".join(table_md_list) if table_md_list else "수집된 표 없음")
        final_content.append("\n\n### [CLEANED TEXT CONTENT]")
        final_content.append(cleaned_text)

        file_path = os.path.join(PROJECT_NAME, "japan_kbeauty_report.txt")
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(final_content))

        print(f"📊 수집 완료: 표 {len(table_md_list)}개 구조화 성공")
        print(f"💾 파일 저장 완료: {file_path}")
        
        # 7. (선택사항) VS Code 터미널에 결과 즉시 출력
        print("\n" + "="*50)
        print(f"추출된 표 개수: {len(table_md_list)}")
        print("="*50)
        # 앞부분만 살짝 확인
        print(cleaned_text + "...")

    else:
        print("❌ 지정한 클래스(tabs-container desc-body)를 찾을 수 없습니다.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")

✅ 타겟 컨텐츠 영역을 찾았습니다.
📊 수집 완료: 표 1개 구조화 성공
💾 파일 저장 완료: Spherical_Insights_Result/japan_kbeauty_report.txt

추출된 표 개수: 1
일본 K-Beauty 제품 시장 통찰력 예측 2035
일본 K-Beauty 제품 시장 규모는 2024년 896.37백만 달러로 예상
시장 규모는 2025년부터 2035년까지 약 8.38%의 CAGR에서 성장할 것으로 예상됩니다.
일본 K-Beauty 제품 시장 크기는 USD 2,173.11에 도달 할 것으로 예상됩니다. 2035년까지 백만
이 보고서에 대한 자세한 내용을 확인하세요 -
무료 샘플 요청
PDF
Spherical Insights & Consulting이 발표한 연구 보고서에 따르면, 일본 K-Beauty 제품 시장은 2035년까지 2,173.11 백만 달러에 도달하여 2025에서 2035년까지 8.38%의 CAGR로 성장했습니다. K-POP과 K-dramas를 포함한 한국 팝컬처의 고급 스킨케어 솔루션과 한국 팝컬처의 영향에 대한 일본 소비자의 성장에 기여하고 있습니다.
시장 개요
일본 k-beauty 제품 시장은 일본 내의 한국 뷰티 및 스킨케어 제품의 수입, 유통 및 판매를 통합합니다. 일본 고객은 K-Beauty 회사의 창조적인 정립과 아주 잘 맞은 고품질 skincare 제품을 위한 강한 친화력이 있습니다. K-Beauty 기술에 필수적 인 시트 마스크 및 혈청의 판매는 다단계 스킨 케어 절차에 대한 일본 소비자의 성장 관심을 반영하는 예를 들어 증가합니다. 일본의 아름다움 의식적인 포뮬러에 매력이 있는 이 트렌드에 의해 실제적이고 분명한 혜택을 제공하는 복잡한 스킨케어 루틴을 향한 소명적인 움직임. K-Beauty는 Cica (Centella Asiatica), 발효 추출물 및 달팽이 매킨을 포함한 혁신적인 재료를 지속적으로 제공 할 수있는 능력 때문에 skincare 부문의 trailblazer입니다. 일본은 특히 노인 인구 때문에

In [3]:
import os
import io
import time
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options

# 1. 환경 설정
PROJECT_NAME = "Spherical_Insights_Final"
IMAGE_DIR = os.path.join(PROJECT_NAME, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

options = Options()
driver = webdriver.Chrome(options=options)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}

# =========================
# 2. 정제 유틸리티 함수
# =========================

def deduplicate_text(text):
    """
    텍스트 내에서 콤마(,)나 줄바꿈으로 반복되는 동일 단어/문구를 하나로 합침
    예: '아모레퍼시픽, 아모레퍼시픽' -> '아모레퍼시픽'
    """
    # 1. 콤마 기준으로 쪼개서 중복 제거 (순서 유지)
    if ',' in text:
        parts = [p.strip() for p in text.split(',')]
        unique_parts = []
        for p in parts:
            if p and p not in unique_parts:
                unique_parts.append(p)
        text = ", ".join(unique_parts)
    
    # 2. 연속된 동일 단어 패턴 제거 (Regex 활용)
    # 똑같은 단어가 바로 다음에 반복될 경우 하나만 남김
    text = re.sub(r'(\b\w+\b)( \1)+', r'\1', text)
    
    return text

def flatten_cols(df):
    """멀티헤더 정리"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in col if 'Unnamed' not in str(c)]).strip() for col in df.columns.values]
    return df

# =========================
# 3. 크롤링 메인 로직
# =========================

url = "https://www.sphericalinsights.com/ko/reports/japan-k-beauty-product-market"

try:
    print(f"🚀 {url} 데이터 수집 및 정제 시작...")
    driver.get(url)
    time.sleep(3)
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    target_area = soup.select_one('div.tabs-container.desc-body')

    if target_area:
        final_content = [f"SOURCE_URL: {url}\n"]
        table_md_list = []

        # [표 추출 및 본문에서 제거]
        table_wrappers = target_area.select('div.table-responsive-outer')
        for t_idx, wrapper in enumerate(table_wrappers, 1):
            table_tag = wrapper.find('table')
            if table_tag:
                try:
                    df = pd.read_html(io.StringIO(str(table_tag)))[0]
                    df = flatten_cols(df)
                    
                    # 표 내부 데이터도 중복 정제 적용
                    df = df.applymap(lambda x: deduplicate_text(str(x)) if isinstance(x, str) else x)
                    
                    md_table = df.to_markdown(index=False)
                    table_md_list.append(f"\n[데이터 표 {t_idx}]\n{md_table}\n")
                    wrapper.decompose() 
                except: continue

        # [텍스트 추출 및 중복 정제 적용]
        # separator를 이용해 의미 단위로 끊어 가져온 뒤 정제
        lines = target_area.get_text(separator='\n', strip=True).split('\n')
        cleaned_lines = []
        for line in lines:
            refined = deduplicate_text(line)
            if refined:
                cleaned_lines.append(refined)
        
        cleaned_body = "\n".join(cleaned_lines)

        # [결과 조립]
        final_content.append("\n### [STRUCTURED TABLES]")
        final_content.append("".join(table_md_list) if table_md_list else "표 없음")
        final_content.append("\n\n### [CLEANED & DEDUPLICATED TEXT]")
        final_content.append(cleaned_body)

        # 파일 저장
        report_path = os.path.join(PROJECT_NAME, "refined_kbeauty_report.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(final_content))

        print(f"✅ 정제 완료: 표 {len(table_md_list)}개, 텍스트 중복 제거 성공")
        print(f"💾 저장 경로: {report_path}")

    else:
        print("❌ 타겟 구역을 찾을 수 없습니다.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")
finally:
    driver.quit()
    print("🏁 크롤링 파이프라인 종료")

🚀 https://www.sphericalinsights.com/ko/reports/japan-k-beauty-product-market 데이터 수집 및 정제 시작...
✅ 정제 완료: 표 0개, 텍스트 중복 제거 성공
💾 저장 경로: Spherical_Insights_Final/refined_kbeauty_report.txt
🏁 크롤링 파이프라인 종료
